## Persona Chatbot With Quality Control

### What Is This Notebook?
This notebook builds a chatbot that answers questions as if it were you. It uses a **context-stuffed prompt with an agentic self-correction loop** — all your profile data is loaded into one big prompt, and a second model checks response quality.

### What Is Gradio?
Gradio is a Python library that creates web-based UIs with minimal code. Instead of building HTML/CSS/JavaScript, you call `gr.ChatInterface(fn=chat).launch()` and get a working chat interface instantly. It handles the web server, message history, and UI — you just provide the chat function.

### What You Will Build
1. Load your profile data from three sources: about me text, LinkedIn PDF, and links
2. Pull extra context from your external links
3. Build a system prompt from all context
4. Create a Gradio chat app
5. Add an evaluator model to check response quality
6. Re-run low-quality answers automatically

Before you begin, make sure these files are updated:
- `me/aboutme.txt`
- `me/linkedin.pdf`
- `me/links.txt`

In [ ]:
# PURPOSE: Install all required Python packages into the notebook environment.
# On macOS + Homebrew Python, --break-system-packages may be needed in notebooks.
%pip install --user --break-system-packages pypdf openai python-dotenv gradio pydantic

In [ ]:
# PURPOSE: Import all libraries needed for PDF reading, web fetching, API calls, and UI.

# Core Python utilities
import io            # Read PDF bytes from URLs as file-like objects
import os            # Access environment variables for API keys
import re            # Parse and clean HTML/text from web pages
from pathlib import Path           # Handle file paths cross-platform
from typing import Any             # Type hints for Gradio history dicts
from urllib.error import HTTPError, URLError  # Catch web fetch errors
from urllib.parse import quote_plus           # URL-encode search queries
from urllib.request import Request, urlopen   # Fetch web pages for context

# Third-party libraries used in this notebook
from dotenv import load_dotenv     # Load API keys from .env file
from openai import OpenAI          # Chat and evaluation API client
from pydantic import BaseModel     # Structured output for evaluator
from pypdf import PdfReader        # Extract text from LinkedIn PDF
import gradio as gr                # Build chat UI with minimal code

In [ ]:
# PURPOSE: Load API keys from .env file and configure OpenAI clients for chat and evaluation.

# Load environment variables from .env file.
load_dotenv(override=True)

# Keep model names in constants so they are easy to change later.
OPENAI_MODEL = "gpt-4o-mini"

# Evaluator model: A second, stronger model that reviews each response for quality.
# Why? The chat model (gpt-4o-mini) is fast and cheap but can produce weak answers.
# The evaluator (gpt-4o) catches issues like hallucinations, off-tone replies, or
# missing context — and triggers a retry if needed. This is the "agentic self-correction loop".
EVALUATOR_MODEL = "gpt-4o"

# Create primary chat client (OpenAI-compatible).
openai = OpenAI()

# Use the same OpenAI client for evaluation (with a different model).
evaluator = openai

print("OpenAI client configured for both chat and evaluation.")

In [ ]:
# PURPOSE: Extract text from your LinkedIn PDF to use as persona context.

# Path() creates a file path object — safer than raw strings for cross-platform support.
PROFILE_PDF_PATH = Path("kb/linkedin.pdf")

# PdfReader opens the PDF file and lets us access its pages.
# str() converts the Path object to a plain string because PdfReader expects that.
reader = PdfReader(str(PROFILE_PDF_PATH))

# Create an empty list to collect text from each page.
# Lists in Python are ordered collections: []
linkedin_sections = []

# Loop through each page in the PDF.
# reader.pages is a list of all pages in the document.
for page in reader.pages:
    # Extract raw text from the current page.
    text = page.extract_text()
    
    # Only add non-empty text. "if text and text.strip()" checks:
    #   1. text exists (not None)
    #   2. text.strip() removes whitespace — if anything remains, it's truthy.
    if text and text.strip():
        # .strip() removes leading/trailing whitespace, then append to our list.
        linkedin_sections.append(text.strip())

# Join all page texts into one string, separated by double newlines.
# "\n\n".join() puts "\n\n" between each item in the list.
linkedin = "\n\n".join(linkedin_sections)

In [ ]:
# PURPOSE: Preview the extracted LinkedIn text to verify it was read correctly.

# f-strings (f"...") let you embed variables inside curly braces {}.
# len() returns the number of characters in a string.
print(f"LinkedIn text length: {len(linkedin)} characters")

# [:1200] is "slicing" — it grabs the first 1200 characters of the string.
# This previews the beginning without printing the whole thing.
print(linkedin[:1200])

In [ ]:
# PURPOSE: Load your short personal introduction from disk.

# Path() creates a cross-platform file path.
ABOUT_ME_PATH = Path("kb/aboutme.txt")

# .read_text() opens the file, reads all content as a string, then closes it.
# encoding="utf-8" ensures special characters (accents, emojis) are read correctly.
about_me = ABOUT_ME_PATH.read_text(encoding="utf-8")

In [ ]:
# PURPOSE: Parse URLs from links.txt and prepare them for web context fetching.

# The persona name your assistant will represent.
# This is a simple string variable.
name = "Rikam Palkar"

# Path to manually curated external links about you.
LINKS_PATH = Path("kb/links.txt")

# Define a function to parse URLs from a text file.
# "def" creates a function. "path: Path" is a type hint (optional but helpful).
# "-> list[str]" means this function returns a list of strings.
def parse_links(path: Path) -> list[str]:
    # If file doesn't exist, return an empty list.
    # .exists() checks if the file is actually there.
    if not path.exists():
        return []
    
    # Create empty list to collect URLs.
    links = []
    
    # Read the file and split into lines.
    # .splitlines() turns "line1\nline2\nline3" into ["line1", "line2", "line3"].
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        # .strip() removes whitespace from start and end of the line.
        line = raw_line.strip()
        
        # Skip empty lines or comment lines (starting with #).
        # "or" means: if either condition is true, skip.
        # .startswith("#") checks if line begins with #.
        if not line or line.startswith("#"):
            continue  # "continue" skips to the next iteration of the loop.
        
        # Use regex (regular expression) to find URLs in the line.
        # re.search() looks for a pattern anywhere in the string.
        # r"https?://\S+" matches "http://" or "https://" followed by non-whitespace.
        match = re.search(r"https?://\S+", line)
        
        # If a URL was found, clean it and add to list.
        if match:
            # .group(0) gets the matched text.
            # .rstrip(")]}.,") removes trailing punctuation that might accidentally be included.
            links.append(match.group(0).rstrip(")]}.,"))
    
    # Remove duplicates while preserving order.
    # dict.fromkeys() creates a dict with links as keys (dicts can't have duplicate keys).
    # list() converts it back to a list.
    return list(dict.fromkeys(links))

# Call the function to get list of URLs from the file.
provided_links = parse_links(LINKS_PATH)

# If no links were found, use a Google search as fallback.
# "not provided_links" is True when the list is empty.
if not provided_links:
    # quote_plus() makes the name URL-safe (spaces become +, etc.).
    provided_links = [f"https://www.google.com/search?q={quote_plus(name)}"]

# Print how many links were loaded and list them all.
print(f"Loaded {len(provided_links)} link(s) for external context:")
for link in provided_links:
    print(f"  - {link}")

In [ ]:
# PURPOSE: Fetch web content from URLs and build the complete system prompt with all context.

# Define a function to fetch text content from a URL.
# Returns the text content or None if fetching fails.
def fetch_url_text(url: str) -> str | None:
    """Attempt to retrieve clean text from a URL."""
    # "try/except" handles errors gracefully.
    # If anything in "try" fails, code jumps to "except".
    try:
        # requests.get() fetches the webpage. timeout=10 means give up after 10 seconds.
        response = requests.get(url, timeout=10)
        
        # raise_for_status() throws an error if the server returned an error (404, 500, etc.).
        response.raise_for_status()
        
        # Get the content type (e.g., "text/html; charset=utf-8").
        content_type = response.headers.get("Content-Type", "")
        
        # Skip non-HTML content (PDFs, images, etc.).
        # "in" checks if the substring exists in the string.
        if "text/html" not in content_type.lower():
            return None
        
        # Parse HTML with BeautifulSoup. "lxml" is a fast parser.
        # This creates a tree structure from the raw HTML.
        soup = BeautifulSoup(response.text, "lxml")
        
        # Remove script and style elements (they contain code, not readable text).
        # .decompose() removes the element from the tree entirely.
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        
        # get_text() extracts all visible text.
        # separator=" " puts spaces between elements.
        # strip=True removes extra whitespace.
        return soup.get_text(separator=" ", strip=True)[:8000]  # Limit to 8000 chars to stay within token limits.
    
    # If anything goes wrong (network error, timeout, etc.), return None.
    except Exception:
        return None


# Define a function to build the complete system prompt with all context.
# This is the "context stuffing" - putting all relevant info into one big prompt.
def build_system_prompt() -> str:
    """Return the persona system prompt, stuffed with all external context."""
    
    # Start building chunks of context to include.
    # A list is like a container that can hold multiple items.
    chunks: list[str] = []
    
    # Add LinkedIn context if we extracted it earlier.
    # "if linkedin_text" is True if the variable exists and isn't empty.
    if linkedin_text:
        chunks.append(f"[LinkedIn profile]\n{linkedin_text}")
    
    # Add aboutme.txt context if we loaded it.
    if aboutme_text:
        chunks.append(f"[About file]\n{aboutme_text}")
    
    # Fetch and add web content from provided links.
    for link in provided_links:
        # Each link gets fetched and added if successful.
        page_text = fetch_url_text(link)
        if page_text:  # Only add if we got content (not None).
            chunks.append(f"[URL: {link}]\n{page_text}")
    
    # Join all chunks with double newlines between them.
    # "\n\n".join() puts "\n\n" between each item in the list.
    external_context = "\n\n".join(chunks)
    
    # Build the final system prompt with instructions and context.
    # Triple quotes allow multi-line strings.
    # .strip() removes leading/trailing whitespace.
    return f"""
You are a digital twin of {name}.
Respond as if you ARE {name}: use first-person ("I", "my"), mirror their tone and expertise, and draw ONLY from the context below.
If the answer isn't in the context, say "I'm not sure – you can ask {name} directly."

### CONTEXT
{external_context}
""".strip()

# Actually build the prompt by calling the function.
system_prompt = build_system_prompt()

# Show how long the prompt is (useful for debugging token limits).
# len() gives the length of a string in characters.
print(f"System prompt length: {len(system_prompt)} characters")

In [ ]:
# PURPOSE: Preview the complete system prompt for debugging (optional).

# This cell just prints the full prompt so you can see what context was loaded.
# Useful to verify all the data made it in correctly.
print(system_prompt)

In [ ]:
# PURPOSE: Define helper to clean chat history and baseline chat function for Gradio.

# Normalize Gradio history so only role/content fields are sent to models.
# Gradio's chat history can include extra metadata that some APIs reject.
# This function cleans it to just the essential fields.
# Input: [
#    {"role": "user", "content": "Hi", "id": "abc123", "timestamp": 1713...},
#    {"role": "assistant", "content": "Hello!", "id": "def456", "metadata": {...}}
# ]
# Output:
# [
#     {"role": "user", "content": "Hi"},
#     {"role": "assistant", "content": "Hello!"}
# ]
def normalize_history(history: list[dict[str, Any]]) -> list[dict[str, str]]:
    # Create empty list for cleaned messages.
    cleaned = []
    
    # Loop through each message in the history.
    # Each message is a "dict" (dictionary) — a key-value data structure.
    for item in history:
        # .get("role", "user") gets the "role" value, or "user" if missing.
        role = item.get("role", "user")
        content = item.get("content", "")
        # Create a new clean dict with only role and content.
        cleaned.append({"role": role, "content": str(content)})
    
    return cleaned


# ============================================================================
# HOW GRADIO CONNECTS TO THIS FUNCTION:
# ============================================================================
# You do NOT call chat() yourself. Gradio calls it automatically!
#
# When you run: gr.ChatInterface(fn=chat).launch()
# The "fn=chat" parameter tells Gradio: "call the chat function when user sends a message"
#
# EXAMPLE FLOW:
# 1. User types "Hello" in browser → Gradio catches this
# 2. Gradio calls: chat("Hello", [])  ← empty history on first message
# 3. Your function returns "Hi! I'm Rikam..."
# 4. Gradio displays that in the chat window
#
# 5. User types "What's your experience?" → Gradio catches this
# 6. Gradio calls: chat("What's your experience?", [
#        {"role": "user", "content": "Hello"},
#        {"role": "assistant", "content": "Hi! I'm Rikam..."}
#    ])
# 7. Your function returns the answer → Gradio displays it
#
# So Gradio is the "caller" — it hooks your function to the web UI.
# ============================================================================

# Baseline chat function (before evaluator loop).
# This is the simplest version — just send message to OpenAI and return response.
# IMPORTANT: This function is called BY GRADIO, not by you directly.
#            Gradio passes (message, history) when user sends a chat message.
def chat(message: str, history: list[dict[str, Any]]) -> str:
    # Clean the history to avoid API errors.
    clean_history = normalize_history(history)
    
    # Build the messages list for the API.
    # OpenAI expects: [system message, ...history..., user message]
    # The + operator combines lists: [a, b] + [c] = [a, b, c]
    messages = ([{"role": "system", "content": system_prompt}]
                + clean_history
                + [{"role": "user", "content": message}])
    
    # Call the OpenAI API.
    # .create() sends the request and waits for the response.
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)
    
    # Extract the text from the response.
    # .choices[0] is the first (usually only) response option.
    # .message.content is the actual text.
    # "or ''" returns empty string if content is None.
    # This returned string is what Gradio displays in the chat window.
    return response.choices[0].message.content or ""

## Special Note For Non-OpenAI Providers

Some providers reject extra metadata fields in chat history.
The `normalize_history()` helper above avoids that by sending only role/content.

In [ ]:
# PURPOSE: Launch the baseline chatbot web UI (without evaluator, for testing).

# ============================================================================
# THIS IS WHERE THE chat FUNCTION GETS CONNECTED TO THE UI:
# ============================================================================
#
# gr.ChatInterface creates a web-based chat UI.
# The key part is: fn=chat
#
# "fn" stands for "function" — it tells Gradio which function to call.
# When you write fn=chat, you're saying:
#   "Hey Gradio, whenever a user sends a message, call my chat() function"
#
# Behind the scenes:
#   1. .launch() starts a web server (usually at http://127.0.0.1:7860)
#   2. User types a message and clicks Send
#   3. Gradio receives the message
#   4. Gradio calls: chat(user_message, conversation_history)
#   5. Your chat() function returns a string
#   6. Gradio displays that string as the bot's response
#
# You never call chat() yourself — Gradio does it for you!
# ============================================================================

gr.ChatInterface(
    fn=chat,  # <-- THE CONNECTION: "call chat() when user sends a message"
    title="Persona Chatbot (Baseline)",
    description="Answers using About Me, LinkedIn, and external links context.",
).launch()  # Starts the web server and opens browser

## Add A Quality-Control Loop

Now we add a second model that evaluates each answer.
If an answer is weak, the assistant rewrites it once using evaluator feedback.

In [ ]:
# PURPOSE: Define a Pydantic schema for structured evaluator output (pass/fail + feedback).

# Pydantic's BaseModel lets us define a schema for the evaluator's response.
# The model will return JSON that must match this structure.
# This is safer than parsing free-form text.
class Evaluation(BaseModel):
    # bool means True or False — did the answer pass quality check?
    is_acceptable: bool
    # str means a text string — why did it pass or fail?
    feedback: str

In [ ]:
# PURPOSE: Build system prompt for the evaluator model with all context for verification.

# This function creates the system prompt for the evaluator model.
# It includes all the same context so the evaluator can verify accuracy.
def build_evaluator_system_prompt(
    name: str,
    about_me: str,
    linkedin: str,
    links: list[str],
    web_context: str,
    ) -> str:
    # Build bullet list of links.
    links_block = "\n".join(f"- {link}" for link in links)
    
    # Return the evaluator's instructions.
    # Note: This is a different role — it's judging answers, not answering questions.
    return f"""You are a strict response evaluator.
You review the assistant's latest answer for quality and faithfulness to persona.
The assistant represents {name} on their website.

Evaluation criteria:
1. Correctness relative to provided context
2. Professional and engaging tone
3. Clarity and relevance to user message
4. No fabricated claims when data is missing

## About Me
{about_me}

## LinkedIn Profile
{linkedin}

## Curated Links
{links_block}

## External Web Context
{web_context}

Return whether the answer is acceptable plus feedback.
"""


# Create the evaluator's system prompt.
evaluator_system_prompt = build_evaluator_system_prompt(
    name=name,
    about_me=about_me,
    linkedin=linkedin,
    links=provided_links,
    web_context=web_context,
)

In [ ]:
# PURPOSE: Build the evaluator's user message from conversation state for review.

# This function builds the "user message" we send to the evaluator.
# It includes all the context needed to judge the response.
def evaluator_user_prompt(reply: str, message: str, history: list[dict[str, str]]) -> str:
    # f-strings with triple quotes can span multiple lines.
    # We're asking the evaluator to review the assistant's reply.
    return (
        "Here is the conversation history:\n\n"
        f"{history}\n\n"
        "Here is the latest user message:\n\n"
        f"{message}\n\n"
        "Here is the assistant's latest response:\n\n"
        f"{reply}\n\n"
        "Decide if the response is acceptable and provide feedback."
    )

In [ ]:
# PURPOSE: Confirm the evaluator model is configured (simple status print).

# Evaluator uses the same OpenAI client with a different model.
print(f"Evaluator enabled using {EVALUATOR_MODEL}.")

In [ ]:
# PURPOSE: Call the evaluator model and return structured pass/fail decision.

# This function sends the response to gpt-4o for quality evaluation.
# It returns an Evaluation object with is_acceptable (True/False) and feedback.
def evaluate(reply: str, message: str, history: list[dict[str, str]]) -> Evaluation:
    # Build the messages for the evaluator API call.
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]

    # Call the evaluator model with "parse" to get structured output.
    # .beta.chat.completions.parse() returns data matching our Evaluation schema.
    # response_format=Evaluation tells the API to return JSON matching our class.
    response = evaluator.beta.chat.completions.parse(
        model=EVALUATOR_MODEL,
        messages=messages,
        response_format=Evaluation,
    )
    
    # .parsed gives us the Evaluation object directly (not raw JSON).
    return response.choices[0].message.parsed

In [ ]:
# PURPOSE: Test the chat model with a sample question before adding retry logic.

# This cell tests the chat model separately to make sure it's working.

# Build a simple test conversation.
messages = [
    {"role": "system", "content": system_prompt},  # The persona instructions.
    {"role": "user", "content": "Do you hold a patent?"},  # A test question.
]

# Call the API and get a response.
response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)

# Extract the text reply.
reply = response.choices[0].message.content or ""

In [ ]:
# PURPOSE: Display the raw model reply from the test call above.

# Just printing the reply variable shows what the model returned.
reply

In [ ]:
# PURPOSE: Run the evaluator on the test reply to verify the evaluation pipeline works.

# This tests the evaluate() function with our test reply.
# Returns an Evaluation object showing if it passed and why.
evaluate(reply, "Do you hold a patent?", messages[:1])

In [ ]:
# PURPOSE: Define retry function that asks the model to improve a rejected answer.

# This is the "self-correction" part of the agentic loop.
# If evaluator rejects an answer, ask the assistant to retry once with feedback.
# We tell the model what went wrong and ask it to try again.
def rerun(reply: str, message: str, history: list[dict[str, str]], feedback: str) -> str:
    # Create an updated system prompt that includes the rejection info.
    # String concatenation with + joins strings together.
    updated_system_prompt = (
        system_prompt
        + "\n\n## Previous answer rejected\n"
        + "Your last answer did not pass quality control.\n"
        + f"## Attempted answer:\n{reply}\n\n"
        + f"## Rejection reason:\n{feedback}\n\n"
        + "Write a better answer now."
    )
    
    # Build messages with the updated prompt.
    messages = ([{"role": "system", "content": updated_system_prompt}]
                + history
                + [{"role": "user", "content": message}])
    
    # Call the API again for a better answer.
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)
    
    return response.choices[0].message.content or ""

In [ ]:
# PURPOSE: Final chat function with full agentic loop: generate → evaluate → retry if needed.

# ============================================================================
# NOTE: This function REPLACES the earlier baseline chat() function.
# It has the same name, so Python will use THIS version instead.
# Gradio will call THIS chat() when you run the final gr.ChatInterface below.
# ============================================================================

# This is the complete "agentic self-correction loop":
# 1. Generate a response with gpt-4o-mini
# 2. Evaluate it with gpt-4o
# 3. If rejected, retry once with the feedback
#
# CALLED BY: Gradio (via fn=chat in gr.ChatInterface)
# NOT called by you directly!
def chat(message: str, history: list[dict[str, Any]]) -> str:
    # Step 1: Clean the history.
    clean_history = normalize_history(history)
    
    # Step 2: Build the messages for the chat model.
    messages = ([{"role": "system", "content": system_prompt}]
                + clean_history
                + [{"role": "user", "content": message}])

    # Step 3: Generate initial response.
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)
    reply = response.choices[0].message.content or ""

    # Step 4: Evaluate the response with the stronger model.
    evaluation = evaluate(reply, message, clean_history)

    # Step 5: If evaluation passed, return the original answer.
    if evaluation.is_acceptable:
        print("Passed evaluation: returning original answer.")
        return reply  # Gradio displays this in the chat window.

    # Step 6: If evaluation failed, retry once with feedback.
    print("Failed evaluation: retrying once with feedback.")
    print(evaluation.feedback)  # Show why it failed.
    improved_reply = rerun(reply, message, clean_history, evaluation.feedback)
    
    return improved_reply  # Gradio displays this improved answer.

In [ ]:
# PURPOSE: Launch the final chatbot UI with quality-control loop enabled.

# This uses the SAME pattern as before:
#   fn=chat tells Gradio to call our chat() function when user sends a message.
#
# But now chat() includes the evaluator loop, so:
#   User sends message → Gradio calls chat() → chat() generates response →
#   chat() evaluates with gpt-4o → if bad, chat() retries → returns final answer →
#   Gradio displays it.

gr.ChatInterface(
    fn=chat,  # <-- Connects to the chat() function defined above (the one with evaluator)
    title="Persona Chatbot (With Evaluator)",
    description="Uses About Me + LinkedIn + links. gpt-4o evaluates responses and retries if needed.",
).launch()